# PML · Lecture 8 (applied) — Bayesian regression on **real data**, and how to *use* the uncertainty

Notebook 1 built the machinery on a synthetic sine curve. Here we point it at a **real medical dataset** and focus on the thing that makes Bayesian regression worth the extra work: **every prediction comes with an honest error bar**, and that error bar is *useful*.

**The data.** `diabetes` (442 patients): ten baseline measurements — age, sex, BMI, blood pressure, and six blood-serum numbers — and a target that quantifies **disease progression one year later**. We'll predict progression, then put the uncertainty to work in four real ways:

1. **An error bar on every prediction** — and a check that it's *trustworthy* (calibration).
2. **Which measurements matter** — coefficients *with* error bars (and honest 'don't know's).
3. **Knowing when the model is out of its depth** — uncertainty that grows off the data.
4. **Deferring the uncertain cases** to a human (selective prediction).

> A prediction without an error bar is a guess. In medicine, engineering, or finance, *how sure* the model is often matters more than the number itself.

Pure `numpy` + `matplotlib` + `scikit-learn` (only to load the dataset) — all preinstalled in Colab.

## 0. Setup — the same Bayesian linear regression from notebook 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def design(X):
    """Add a bias column of 1s: raw features -> design matrix [1 | X]."""
    X = np.atleast_2d(X)
    return np.column_stack([np.ones(len(X)), X])

def blr_posterior(Phi, y, alpha, beta):
    M = Phi.shape[1]
    S_N = np.linalg.inv(alpha*np.eye(M) + beta * Phi.T @ Phi)
    m_N = beta * S_N @ Phi.T @ y
    return m_N, S_N

def predictive(Phi_star, m_N, S_N, beta):
    mean = Phi_star @ m_N
    epistemic = np.sum((Phi_star @ S_N) * Phi_star, axis=1)   # phi^T S_N phi (model uncertainty)
    var = 1.0/beta + epistemic                               # + noise floor
    return mean, np.sqrt(var), np.sqrt(epistemic)

## 1. Load and look at the real data

Always look first. BMI alone already trends with progression — but it's noisy, so any single prediction should come with a range, not a point.

In [ ]:
data = load_diabetes()
X_raw, y_raw, names = data.data, data.target, data.feature_names
print('patients:', X_raw.shape[0], ' features:', names)
print('target (progression): min %.0f  mean %.0f  max %.0f' % (y_raw.min(), y_raw.mean(), y_raw.max()))

bmi = X_raw[:, names.index('bmi')]
plt.scatter(bmi, y_raw, s=12, alpha=0.6)
plt.xlabel('BMI (standardized)'); plt.ylabel('disease progression')
plt.title('Real data is noisy: BMI vs progression'); plt.show()

## 2. Preprocess → design matrix

**Standardize** every feature (mean 0, unit variance) so the weights are comparable and the prior treats them evenly; standardize the target too so the noise scale is $O(1)$. Then split into train/test and build the design matrix $\Phi=[\mathbf 1\mid X]$.

In [ ]:
idx = rng.permutation(len(y_raw))
tr, te = idx[:350], idx[350:]                 # 350 train / 92 test

# standardize using TRAIN statistics only (never peek at the test set)
Xmean, Xstd = X_raw[tr].mean(0), X_raw[tr].std(0)
y_mean, y_std = y_raw[tr].mean(), y_raw[tr].std()
Xs = (X_raw - Xmean) / Xstd
ys = (y_raw - y_mean) / y_std
Phi_tr, Phi_te = design(Xs[tr]), design(Xs[te])
print('design matrix Phi_tr:', Phi_tr.shape, ' (350 patients x 11 columns: bias + 10 features)')

## 3. Fit the Bayesian model

We need the noise precision $\beta=1/\sigma^2$. A clean estimate: fit ordinary least squares, look at the residual variance, and set $\beta=1/\widehat{\sigma}^2$. Use a mild prior $\alpha=1$.

In [ ]:
w_ols, *_ = np.linalg.lstsq(Phi_tr, ys[tr], rcond=None)
resid = ys[tr] - Phi_tr @ w_ols
beta  = 1.0 / resid.var(ddof=Phi_tr.shape[1])     # 1 / estimated noise variance
alpha = 1.0
m_N, S_N = blr_posterior(Phi_tr, ys[tr], alpha, beta)
print('estimated noise sigma (standardized units): %.2f' % (1/np.sqrt(beta)))

## Use 1 — an error bar on every prediction, and is it *trustworthy*?

Predict each held-out patient with a mean and a $\pm 2\sigma$ interval (converted back to real progression units). Then the crucial check: **calibration** — do about 95% of the true values actually land inside their 95% intervals? If yes, the error bars mean what they say.

In [ ]:
mean_s, sd_s, epi_s = predictive(Phi_te, m_N, S_N, beta)
pred = mean_s * y_std + y_mean            # back to real progression units
sd   = sd_s   * y_std
true = y_raw[te]

coverage = np.mean(np.abs(true - pred) <= 2*sd)
print(f'95% interval coverage on held-out patients: {coverage:.1%}  (target ~95%)')

o = np.argsort(pred)
plt.errorbar(np.arange(len(te)), pred[o], yerr=2*sd[o], fmt='o', ms=3,
             elinewidth=1, capsize=2, alpha=0.7, label='prediction ±2σ')
plt.scatter(np.arange(len(te)), true[o], c='k', s=10, zorder=3, label='actual')
plt.xlabel('held-out patients (sorted by prediction)'); plt.ylabel('progression')
plt.legend(); plt.title('Every prediction carries a 95% interval'); plt.show()

The coverage comes out near 95% — so when the model reports $\pm 2\sigma$, the truth really is inside about 19 times out of 20. **That** is what makes the uncertainty usable: a *narrow* bar is a prediction you can act on; a *wide* bar says "get more information before deciding."

## Use 2 — which measurements matter, *with error bars on the answer*

The posterior is a full Gaussian over the weights, so each coefficient has its own uncertainty $\pm 2\sqrt{(S_N)_{jj}}$. A weight whose interval **excludes 0** is a feature the model is confident about; one whose interval **straddles 0** is an honest "don't know."

In [ ]:
w   = m_N[1:]                       # drop the bias
werr = 2*np.sqrt(np.diag(S_N))[1:]
sig = np.abs(w) > werr              # 2-sigma away from zero

colors = ['C0' if s else '0.7' for s in sig]
plt.bar(names, w, yerr=werr, color=colors, capsize=3)
plt.axhline(0, color='k', lw=0.8)
plt.ylabel('posterior weight ± 2σ'); plt.title('Which features matter (blue = confidently ≠ 0)')
plt.show()
print('confidently non-zero:', [n for n, s in zip(names, sig) if s])

**BMI, s5, blood pressure and sex** stand out — the model is sure they matter. The serum markers **s1 and s2** get *enormous* error bars: they're strongly correlated, so the model can't tell their individual effects apart and says so. A plain point estimate would hide that; the Bayesian error bar surfaces it — real, honest interpretability. (These are *associations* in this dataset, not proven causes.)

## Use 3 — knowing when the model is out of its depth

Split the predictive variance into the fixed **noise floor** $1/\beta$ and the **epistemic** part $\phi(x_\ast)^\top S_N\,\phi(x_\ast)$ — the model's *own* uncertainty, which grows for inputs unlike the training data. Below we plot that **epistemic** band (the noise floor is a constant added on top); fit BMI alone and predict across a range that runs off the edge of the observed data.

In [ ]:
# 1-D Bayesian fit on BMI only, so we can see the band
b = Xs[:, names.index('bmi')]
Pb = design(b[tr, None]);  mb, Sb = blr_posterior(Pb, ys[tr], alpha, beta)

grid = np.linspace(b.min()-1.5, b.max()+1.5, 200)     # extend past the data
gmean, gsd, gepi = predictive(design(grid[:, None]), mb, Sb, beta)
gmean = gmean*y_std + y_mean
gepi  = gepi*y_std          # EPISTEMIC (model) uncertainty, in real units

plt.axvspan(b.min(), b.max(), color='k', alpha=0.05, label='observed BMI range')
plt.scatter(b, y_raw, s=10, alpha=0.4)
plt.plot(grid, gmean, 'b', label='predictive mean')
plt.fill_between(grid, gmean-2*gepi, gmean+2*gepi, color='b', alpha=0.2,
                 label='±2σ model (epistemic) uncertainty')
plt.xlabel('BMI (standardized)'); plt.ylabel('progression'); plt.legend()
plt.title('Model uncertainty grows off the data — refuse to extrapolate silently'); plt.show()

The **model-uncertainty band flares** once BMI leaves the range the model has actually seen (the full prediction interval adds the constant measurement-noise floor on top of this). In deployment you'd **flag** any patient sitting in that high-epistemic-uncertainty zone: the model is extrapolating and shouldn't be trusted blindly.

## Use 4 — defer the uncertain cases (selective prediction)

A safe, standard pattern: let the model **auto-decide** the cases it's most confident about and **hand the rest to a human**. Rank the held-out patients by the model's epistemic uncertainty and compare the error on the confident half vs the uncertain half.

In [ ]:
epi = epi_s * y_std                       # epistemic std in real units
rank = np.argsort(epi)
lo, hi = rank[:len(rank)//2], rank[len(rank)//2:]
rmse = lambda mask: np.sqrt(np.mean((true[mask] - pred[mask])**2))
print(f'RMSE on the confident half (auto-decide): {rmse(lo):.1f}')
print(f'RMSE on the uncertain half (defer):       {rmse(hi):.1f}')
print('-> the confident half is modestly more accurate; deferring the least-certain cases')
print('   to a human is a safe, standard way to deploy an uncertain model.')

## Your turn

1. **Prior strength.** Re-fit with `alpha` in `[0.01, 1, 100]`. What happens to the feature error bars and to the calibration coverage?
2. **Fewer patients.** Train on only the first 60 patients. Do the intervals get wider? Does coverage stay near 95%?
3. **A confident vs an unsure patient.** Print the test patient with the smallest and the largest predictive `sd`. What's different about them?
4. **Basis functions.** In Use 3, swap BMI's linear fit for a small Gaussian-RBF basis (reuse notebook 1's `rbf_design`). Does the curve bend? Does the band still flare off-data?

## Recap — *why* the error bar earns its keep
- **Calibrated** intervals (~95% coverage) turn a number into a *trustworthy* number.
- **Coefficient** error bars separate "this matters" from "can't tell" — real interpretability.
- **Epistemic** uncertainty flags **extrapolation** — the model knows what it doesn't know.
- **Selective prediction** routes the shaky cases to a human — a safe way to deploy.

That is the payoff of going Bayesian: not just a fit, but a fit that tells you *how much to trust it*.